---
title: "Part 12: Presenting Data with Great Tables"
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sambaiga/ds-mlops-path/blob/main/tutorials/01-python-basics/12-great-tables.ipynb) [![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue.svg?logo=jupyter&logoColor=white)](https://raw.githubusercontent.com/sambaiga/ds-mlops-path/main/tutorials/01-python-basics/12-great-tables.ipynb)


**DS-MLOps Data Analysis**

**Python 3.12+ | Author: Anthony Faustine**

## Before you begin

This notebook assumes you have completed Parts 8 and 9 (pandas core and operations). All examples use the same student exam dataset: 2,400 students, marks in three subjects, course metadata.

Part 13 (`13-polars.ipynb`) continues with Polars, the modern DataFrame alternative.

| Top

::: {.callout-note collapse="true" title="Learning Objectives"}
## Learning Objectives

| # | Skill | Covered in |
|---|---|---|
| 1 | Explain when a table communicates better than a chart | Sec. 1 |
| 2 | Build a branded table with `themed_gt()` and `GT` | Sec. 2 |
| 3 | Format numbers, percentages, and missing values | Sec. 3 |
| 4 | Add column spanners and relabel columns | Sec. 4 |
| 5 | Highlight best and worst values with conditional styling | Sec. 5 |
| 6 | Add summary rows and source notes | Sec. 6 |
| 7 | Use `metrics_report()` for model comparison tables | Sec. 7 |
:::


In [ ]:
from great_tables import GT, html, loc, md as gt_md, style
import numpy as np
import pandas as pd

from ark.plot.gt_style import metrics_report, themed_gt

df = pd.read_csv("data/university_analytics.csv")
df["average_marks"] = (df["midterm_score"] + df["final_score"] + df["project_score"]) / 3
df.head(3)

## 1. When Tables Beat Charts

A chart compresses a distribution into shape. A table preserves exact values — it is the right tool when a reader needs to look up a specific number, compare a small set of precise quantities, or see a ranking with exact margins.

Use a table when:
- The audience needs to read individual values (not just spot the trend)
- You are comparing across two dimensions simultaneously (rows × columns)
- Precision matters more than pattern (e.g., a leaderboard, a metrics report)

<div class='ark-concept'>
<span class='ark-callout-title'><i class="bi bi-info-circle-fill"></i> Key Concept: GT wraps a DataFrame, it does not copy it</span><br><br>
<code>GT(df)</code> takes a reference to your existing DataFrame and wraps it with display logic. Every method — <code>.tab_header()</code>, <code>.fmt_number()</code>, <code>.tab_style()</code> — returns a new <code>GT</code> object. Chain them like you chain pandas. Nothing is mutated.
</div>

## 2. Your First Styled Table

Raw `df.head()` output is readable in a notebook but looks like a spreadsheet dump in a rendered report. `GT` turns the same data into a publication-ready table in a few lines, and `themed_gt()` applies the project brand on top.

In [ ]:
# Summarise mean scores and fail rate by gender
summary = (
    df.groupby("gender")
    .agg(
        n_students=("student_id", "count"),
        midterm=("midterm_score", "mean"),
        final=("final_score", "mean"),
        project=("project_score", "mean"),
        fail_rate=("passed", lambda x: (~x).mean()),
    )
    .reset_index()
    .round(2)
)
summary

In [ ]:
# Wrap with GT, add a title, apply brand theme
table = (
    GT(summary)
    .tab_header(
        title=gt_md("**Mean Exam Marks by Gender**"),
        subtitle="Students with complete mark records — all three subjects",
    )
    .tab_source_note("Source: DS-MLOps student dataset · 2,400 rows")
)
themed_gt(table, n_rows=len(summary))

<div class='ark-activity'>
<span class='ark-callout-title'><i class="bi bi-puzzle-fill"></i> Activity 1 - Aggregate by Caste</span><br><br>
<b>Goal:</b> Build the same summary grouped by <code>caste</code> instead of <code>gender</code>, then wrap it with <code>themed_gt()</code>. How many caste categories are there?
<pre>summary_caste = (
    df.groupby("caste")
    .agg(...)
    .reset_index()
)
themed_gt(GT(summary_caste).tab_header(title="..."), n_rows=len(summary_caste))</pre>
</div>

## 3. Formatting Numbers, Percentages, and Missing Values

The default float rendering in GT is verbose — eight decimal places for a mark that only needs two. `fmt_*` methods fix this column by column.

In [ ]:
formatted = (
    GT(summary)
    .tab_header(
        title=gt_md("**Mean Exam Scores by Gender**"),
        subtitle="All figures rounded to one decimal place",
    )
    .cols_label(
        gender="Gender",
        n_students="Students",
        midterm="Midterm",
        final="Final",
        project="Project",
        fail_rate="Fail Rate",
    )
    .fmt_integer(columns="n_students")
    .fmt_number(columns=["midterm", "final", "project"], decimals=1)
    .fmt_percent(columns="fail_rate", decimals=1)
    .tab_source_note("Source: DS-MLOps university analytics dataset · 2,400 rows")
)
themed_gt(formatted, n_rows=len(summary))

<div class='ark-protip'>
<span class='ark-callout-title'><i class="bi bi-lightbulb-fill"></i> Pro Tip: <code>fmt_missing()</code> replaces NaN before any other format runs</span><br><br>
If a cell is <code>NaN</code>, <code>fmt_number()</code> will render it as <code>nan</code>. Call <code>.fmt_missing(columns=..., missing_text="—")</code> before your numeric formats so missing cells get a clean dash instead.
</div>

<div class='ark-activity'>
<span class='ark-callout-title'><i class="bi bi-puzzle-fill"></i> Activity 2 - School Summary with Missing Values</span><br><br>
<b>Goal:</b> Aggregate by <code>course</code> and include <code>total_toilets</code> (which has missing values). Format the toilet count with <code>fmt_missing(missing_text="—")</code> before formatting it as an integer.
<pre>course_summary = (
    df.groupby("course")
    .agg(
        students=("student_id", "count"),
        avg_maths=("midterm_score", "mean"),
        toilets=("total_toilets", "first"),
    )
    .reset_index()
)
# Hint: chain .fmt_missing(columns="toilets") before .fmt_integer()</pre>
</div>

## 4. Column Spanners

When columns naturally group together — three subject marks, two teacher counts — a spanner label ties them visually without repeating words in every column header.

In [ ]:
# Course performance table: one row per course
course_detail = (
    df.groupby("course")
    .agg(
        students=("student_id", "count"),
        midterm=("midterm_score", "mean"),
        final=("final_score", "mean"),
        project=("project_score", "mean"),
        pass_rate=("passed", "mean"),
    )
    .reset_index()
    .round(2)
)
course_detail

<div class='ark-activity'>
<span class='ark-callout-title'><i class="bi bi-puzzle-fill"></i> Activity 3 - Add a Dropout Spanner</span><br><br>
<b>Goal:</b> Add <code>fail_rate</code> to <code>course_detail</code> and group it under a new spanner labelled <code>"Outcomes"</code>.
<pre># Hint: add fail_rate to the .agg() call above, then:
.tab_spanner(label="Outcomes", columns=["fail_rate"])
.fmt_percent(columns="fail_rate", decimals=1)</pre>
</div>

## 5. Conditional Styling — Highlighting Best and Worst

Stripe alternating rows for readability. Highlight the best value in each subject column so a reader can spot the top course at a glance.

In [ ]:
from ark.plot.colors import INFO, PRIMARY, SUCCESS, SURFACE_MUTED, WHITE

highlighted = (
    GT(course_detail)
    .tab_header(title=gt_md("**Course Performance — Top Courses Highlighted**"))
    .cols_label(
        course="Course",
        students="Students",
        midterm="Midterm",
        final="Final",
        project="Project",
        pass_rate="Pass Rate",  # noqa: S106
    )
    .fmt_integer(columns="students")
    .fmt_number(columns=["midterm", "final", "project"], decimals=1)
    .fmt_percent(columns="pass_rate", decimals=1)
    .tab_style(
        style=style.fill(color=SUCCESS),
        locations=loc.body(
            columns="pass_rate",
            rows=lambda df_gt: df_gt["pass_rate"] == df_gt["pass_rate"].max(),
        ),
    )
    .tab_style(
        style=style.fill(color="#FEF2F2"),
        locations=loc.body(
            columns="pass_rate",
            rows=lambda df_gt: df_gt["pass_rate"] == df_gt["pass_rate"].min(),
        ),
    )
)
themed_gt(highlighted, n_rows=len(course_detail))

<div class='ark-concept'>
<span class='ark-callout-title'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>loc.body()</code> targets individual cells</span><br><br>
<code>loc.body(rows=..., columns=...)</code> accepts integer indices, column names, or boolean masks for <code>rows</code>. You can pass a list of row indices to highlight multiple cells at once, or omit <code>rows</code> entirely to style an entire column.
</div>

<div class='ark-activity'>
<span class='ark-callout-title'><i class="bi bi-puzzle-fill"></i> Activity 4 - Highlight the Worst Value</span><br><br>
<b>Goal:</b> Add styling to highlight the <em>lowest</em> maths score (use a red fill instead of the slate fill).
<pre>from ark.plot.colors import DANGER
worst_row = int(course_detail["maths"].idxmin())
highlighted = highlighted.tab_style(
    style=[style.fill(color="#FEF2F2"), style.text(weight="bold", color=DANGER)],
    locations=loc.body(rows=worst_row, columns="maths"),
)</pre>
</div>

## 6. Summary Rows

Grand summary rows are the final row that aggregates the whole table — mean, total, or count. They render separately from the body so they always stay visible even if you add row groups.

In [ ]:
from great_tables import vals as gt_vals

with_summary = (
    GT(course_detail)
    .tab_header(title=gt_md("**Course Summary with Grand Mean**"))
    .cols_label(
        course="Course",
        students="Students",
        midterm="Midterm",
        final="Final",
        project="Project",
        pass_rate="Pass Rate",  # noqa: S106
    )
    .tab_spanner(label="Score (0-100)", columns=["midterm", "final", "project"])
    .fmt_integer(columns="students")
    .fmt_number(columns=["midterm", "final", "project"], decimals=1)
    .fmt_percent(columns="pass_rate", decimals=1)
    .grand_summary_rows(
        columns=["midterm", "final", "project"],
        fns={"Mean": lambda x: x.mean()},
        fmt=lambda x: GT.fmt_number(x, decimals=1),
    )
)
themed_gt(with_summary, n_rows=len(course_detail))

<div class='ark-protip'>
<span class='ark-callout-title'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Row groups let you fold related rows under a shared label</span><br><br>
<code>.tab_row_group(label="Top Schools", rows=idx_list)</code> clusters rows under a collapsible group header. Combine with <code>.row_group_order(groups=[...])</code> to control the order groups appear.
</div>

<div class='ark-activity'>
<span class='ark-callout-title'><i class="bi bi-puzzle-fill"></i> Activity 5 - Add a Total Students Row</span><br><br>
<b>Goal:</b> Add a second summary function that totals the <code>students</code> column alongside the existing grand mean for marks.
<pre>.grand_summary_rows(
    columns=["maths", "english", "science", "students"],
    fns={
        "Grand Mean": lambda x: x.mean(),
        "Total": lambda x: x.sum(),
    },
    fmt=lambda x: x.fmt_number(decimals=3),
)</pre>
</div>

## 7. Model Comparison with `metrics_report()`

The `ark.plot.gt_style` module ships `metrics_report()` — a one-liner that produces a publication-ready comparison table, highlights the best value in each column, and applies the brand theme. Use it whenever you need to compare model runs, hyperparameter sweeps, or cross-validation fold summaries.

In [ ]:
# Simulate a model comparison table
comparison = pd.DataFrame(
    {
        "Model": ["Linear Regression", "Ridge (α=0.1)", "Ridge (α=1.0)", "Random Forest"],
        "MAE": [0.0821, 0.0809, 0.0798, 0.0743],
        "RMSE": [0.1042, 0.1031, 0.1019, 0.0961],
        "R2": [0.7812, 0.7841, 0.7876, 0.8103],
    }
)

metrics_report(
    comparison,
    metrics=["MAE", "RMSE", "R2"],
    minimize_cols=["MAE", "RMSE"],
    maximize_cols=["R2"],
    title="Regression Model Comparison",
    subtitle="Student exam score prediction · hold-out test set",
    source_note="All metrics computed on the same 20% hold-out split",
)

<div class='ark-concept'>
<span class='ark-callout-title'><i class="bi bi-info-circle-fill"></i> Key Concept: <code>metrics_report</code> is a thin wrapper — you can extend it</span><br><br>
<code>metrics_report()</code> calls <code>themed_gt()</code> internally and returns a <code>GT</code> object. You can keep chaining on top: <code>metrics_report(...).tab_spanner(...)</code> or <code>.cols_width({"Model": "35%"})</code> all work because the return type is just a standard <code>GT</code>.
</div>

<div class='ark-activity'>
<span class='ark-callout-title'><i class="bi bi-puzzle-fill"></i> Activity 6 - Add a Cross-Validation Table</span><br><br>
<b>Goal:</b> Create a 5-fold cross-validation results table for two models (your choice of MAE / R² values) and render it with <code>metrics_report()</code>.
<pre>cv_results = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "Model A MAE": [...],
    "Model B MAE": [...],
})
metrics_report(cv_results, metrics=["Model A MAE", "Model B MAE"],
               minimize_cols=["Model A MAE", "Model B MAE"])</pre>
</div>

## Summary

| GT method | What it does |
|---|---|
| `GT(df)` | Wrap a DataFrame — start here |
| `.tab_header(title, subtitle)` | Add a title block above the table |
| `.cols_label(**mapping)` | Rename columns for display |
| `.fmt_number(columns, decimals)` | Format floats to fixed precision |
| `.fmt_integer(columns)` | Format whole numbers with thousands separator |
| `.fmt_percent(columns, decimals)` | Format 0–1 floats as `12.3 %` |
| `.fmt_missing(missing_text)` | Replace `NaN` with a display string |
| `.tab_spanner(label, columns)` | Group columns under a shared header |
| `.tab_style(style, locations)` | Apply CSS to specific cells |
| `.grand_summary_rows(fns)` | Add an aggregate row at the bottom |
| `themed_gt(gt, n_rows)` | Apply project brand — call this last |
| `metrics_report(df, ...)` | One-liner model comparison table |

Next: `13-polars.ipynb` — fast DataFrames with Polars.